# Promo Finder: microdrama screenplay → ranked promos
Reads every episode screenplay and returns ranked **20–60s promo clips**: episode, start/end of each segment, the hook, the cliffhanger and one line on why it works.

**How it works**
1. **Rules (no AI):** finds every clean cut point (camera-shot boundaries where nobody is mid-sentence) and scores every 20–60s window on hook / cliffhanger / standalone. Runs free and instantly; it's the baseline and the safety net.
2. **AI per episode:** the model reads the numbered screenplay and proposes promos by **line ID**, never by typing timestamps.
3. **AI cross-episode montages:** joins 2–4 key moments from different episodes.
4. **Code validation:** every AI pick is snapped to clean cuts and forced to 20–60s; bad picks are logged in `rejected.json`.
5. **Blind AI judge:** a second pass re-scores the top picks seeing only what a viewer would see.
6. **Rank:** quality + small bonus for early episodes, minus spoiler risk, max 2 promos per episode in the top 10.

**Run:** Runtime → Run all. You'll be asked to upload the screenplay zip and for an API key (free Gemini key: https://aistudio.google.com/apikey).

## 1. Code (embedded; same as the GitHub repo)

In [ ]:
!mkdir -p promo_finder && pip -q install requests pandas

In [ ]:
%%writefile promo_finder/__init__.py
"""Promo Finder: find and rank promo clips in microdrama screenplays."""

In [ ]:
%%writefile promo_finder/parser.py
"""Parse Kuku-style microdrama screenplays into timestamped events.

Each episode file looks like:
    EPISODE NUMBER: Episode 1
    EPISODE LOGLINE: ...
    --- SCRIPT START ---
    SCENE 01
    Scene heading: INT. CATHEDRAL – DAY
    Scene start: 00:00.00
    Scene end: 02:18.00
    **CAMERA:**            /  **ACTION:**  /  **SOUND FX:**  /  **NAME (Delivery):**
    Start: (00:00.00) | End: (00:04.80)
    text...
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field, asdict
from pathlib import Path

TIME_RE = r"(\d+):(\d{2}(?:\.\d+)?)"
BLOCK_RE = re.compile(r"^\*\*(.+?):\*\*\s*$")
SPAN_RE = re.compile(rf"^Start:\s*\({TIME_RE}\)\s*\|\s*End:\s*\({TIME_RE}\)")
NON_SPEAKERS = {"CAMERA": "camera", "ACTION": "action", "SOUND FX": "sound", "SOUND": "sound", "SFX": "sound"}


def to_sec(m: str, s: str) -> float:
    return int(m) * 60 + float(s)


def fmt(t: float) -> str:
    """Seconds -> 'mm:ss.ss' (same style as the screenplays)."""
    t = max(0.0, t)
    return f"{int(t // 60):02d}:{t % 60:05.2f}"


def parse_time(s: str) -> float:
    m = re.fullmatch(r"\s*" + TIME_RE + r"\s*", str(s))
    if not m:
        raise ValueError(f"bad timestamp: {s!r}")
    return to_sec(m.group(1), m.group(2))


@dataclass
class Event:
    idx: int
    kind: str          # camera | action | dialogue | sound
    start: float
    end: float
    text: str
    scene: int
    speaker: str = ""
    delivery: str = ""

    def line(self) -> str:
        """Compact one-line form used in prompts and reports."""
        span = f"[{fmt(self.start)}-{fmt(self.end)}]"
        if self.kind == "dialogue":
            d = f" ({self.delivery})" if self.delivery else ""
            return f"{span} {self.speaker}{d}: {self.text}"
        return f"{span} {self.kind.upper()}: {self.text}"


@dataclass
class Episode:
    number: int
    title: str
    logline: str
    continuity: str
    events: list[Event] = field(default_factory=list)
    scenes: list[dict] = field(default_factory=list)
    source: str = ""

    @property
    def duration(self) -> float:
        ends = [e.end for e in self.events] + [s["end"] for s in self.scenes]
        return max(ends) if ends else 0.0

    def transcript(self, kinds=("camera", "action", "dialogue", "sound")) -> str:
        return "\n".join(e.line() for e in self.events if e.kind in kinds)

    def to_dict(self) -> dict:
        d = asdict(self)
        d["duration"] = self.duration
        return d


def _split_speaker(label: str) -> tuple[str, str]:
    # "DETECTIVE MIA (Calm, Firm)" -> ("DETECTIVE MIA", "Calm, Firm")
    # "MATRIARCH (O.S.) (Firm)" -> ("MATRIARCH (O.S.)", "Firm")
    m = re.match(r"^(.*?)\s*\(([^()]*)\)\s*$", label)
    if m and m.group(2).upper() not in {"O.S.", "V.O.", "CONT'D"}:
        return m.group(1).strip(), m.group(2).strip()
    return label.strip(), ""


def parse_episode(path: str | Path) -> Episode:
    path = Path(path)
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    header = {}
    for ln in lines[:10]:
        if ":" in ln and ln.split(":", 1)[0].isupper():
            k, v = ln.split(":", 1)
            header[k.strip()] = v.strip()
    num_m = re.search(r"(\d+)", header.get("EPISODE NUMBER", "")) or re.search(r"episode-(\d+)", path.name)
    ep = Episode(
        number=int(num_m.group(1)) if num_m else 0,
        title=header.get("EPISODE NUMBER", path.stem),
        logline=header.get("EPISODE LOGLINE", ""),
        continuity=header.get("CONTINUITY NOTES", ""),
        source=path.name,
    )

    scene_no, i = 0, 0
    while i < len(lines):
        ln = lines[i].strip()
        if re.match(r"^SCENE\s+\d+", ln):
            scene_no = int(re.findall(r"\d+", ln)[0])
            ep.scenes.append({"scene": scene_no, "heading": "", "start": 0.0, "end": 0.0})
        elif ln.startswith("Scene heading:") and ep.scenes:
            ep.scenes[-1]["heading"] = ln.split(":", 1)[1].strip()
        elif ln.startswith("Scene start:") and ep.scenes:
            ep.scenes[-1]["start"] = parse_time(ln.split(":", 1)[1])
        elif ln.startswith("Scene end:") and ep.scenes:
            ep.scenes[-1]["end"] = parse_time(ln.split(":", 1)[1])
        else:
            bm = BLOCK_RE.match(ln)
            if bm and i + 1 < len(lines):
                sm = SPAN_RE.match(lines[i + 1].strip())
                if sm:
                    label = bm.group(1).strip()
                    start, end = to_sec(sm.group(1), sm.group(2)), to_sec(sm.group(3), sm.group(4))
                    j, body = i + 2, []
                    while j < len(lines) and lines[j].strip() and not BLOCK_RE.match(lines[j].strip()):
                        body.append(lines[j].strip())
                        j += 1
                    text = " ".join(body).strip()
                    kind = NON_SPEAKERS.get(label.upper())
                    if kind:
                        ev = Event(len(ep.events), kind, start, end, text, scene_no)
                    else:
                        spk, deliv = _split_speaker(label)
                        ev = Event(len(ep.events), "dialogue", start, end, text.strip('"“” '), scene_no, spk, deliv)
                    ep.events.append(ev)
                    i = j
                    continue
        i += 1
    ep.events.sort(key=lambda e: (e.start, e.end))
    for k, e in enumerate(ep.events):
        e.idx = k
    return ep


def load_show(folder: str | Path) -> list[Episode]:
    """Load every *.txt screenplay under a folder (recursively), sorted by episode number."""
    files = sorted(Path(folder).rglob("*.txt"))
    eps = [parse_episode(f) for f in files]
    eps = [e for e in eps if e.events]
    return sorted(eps, key=lambda e: e.number)

In [ ]:
%%writefile promo_finder/rules.py
"""Plain-code rules: legal cut points, validation/repair of promos, and a
no-AI heuristic scorer used (a) as a baseline and (b) as a second opinion on
AI picks.

Design rule: the AI is never trusted with timestamps. Every segment it
proposes is snapped to a real camera cut where no line of dialogue is
running, and the 20-60s length is enforced here, not in the prompt.
"""
from __future__ import annotations

import re
from .parser import Episode, Event, fmt

MIN_LEN, MAX_LEN, HOOK_WINDOW = 20.0, 60.0, 3.0
SNAP_TOL = 3.0  # seconds a proposed timestamp may move to reach a clean cut

# --- keyword lists (deliberately small and readable; tune per genre) -------
PHYSICAL = r"slap|strike|struck|punch|kiss|gasp|scream|collaps|blood|knife|dagger|sword|blade|grab|seiz|shove|crash|" \
           r"shatter|slam|throw|hurl|chok|poison|arsenic|faint|one knee|kneel|gun|burn|stab|wound|bleed|drag|unmask|" \
           r"rip|tear[s]? (?:open|off)|guards? (?:seize|grab)|fall|plunge|smash|whip|attack|lunge|pin[s]? her"
STRONG_WORDS = r"marry|married|wife|husband|divorce|pregnan|child|dead|die|kill|murder|poison|liar|lie|betray|never|" \
               r"how dare|guards|arrest|duke|duchess|secret|truth|love|hate|reveng|ruin|worthless|disgrace|mistress|" \
               r"heir|title|blood|traitor|prison|hang|exile|cheat|steal|stole|enough|kneel|beg|get out|who are you"
INTENSE_DELIVERY = r"furious|scream|shout|shock|horrif|enrag|desperat|menac|gloat|venom|outrag|sob|crying|roar|threat|" \
                   r"cold|cruel|livid|panick|terrif|accus|icy|deadly|explod|seething|hiss"
CLIFF_SOUND = r"cliffhanger|sting|drone|swell|chord|ominous|thud|crash|boom|silence|heartbeat"
REPLY_STARTERS = r"^(yes|no|and|but|so|because|then|well|oh|why\??|what\??|really\??|of course)\b"
DULL_CAMERA = r"establishing|wide shot|aerial|exterior|drone shot|landscape"

_rx = lambda p: re.compile(p, re.I)
PHYSICAL_RX, STRONG_RX, INTENSE_RX = _rx(PHYSICAL), _rx(STRONG_WORDS), _rx(INTENSE_DELIVERY)
CLIFF_RX, REPLY_RX, DULL_RX = _rx(CLIFF_SOUND), _rx(REPLY_STARTERS), _rx(DULL_CAMERA)


# --- cut points ------------------------------------------------------------
def cut_points(ep: Episode) -> list[float]:
    """Times where an editor can cleanly cut: camera-shot boundaries and scene
    boundaries, excluding any moment where a dialogue line is being spoken."""
    pts = {0.0, round(ep.duration, 2)}
    for e in ep.events:
        if e.kind == "camera":
            pts.update({round(e.start, 2), round(e.end, 2)})
    for s in ep.scenes:
        pts.update({round(s["start"], 2), round(s["end"], 2)})
    talking = [(e.start, e.end) for e in ep.events if e.kind == "dialogue"]
    ok = [t for t in sorted(pts) if not any(a + 0.05 < t < b - 0.05 for a, b in talking)]
    return ok


def snap(t: float, cuts: list[float], tol: float = SNAP_TOL, prefer: str = "near") -> float | None:
    """Move t to a clean cut. prefer='before' for starts (never skip the hook line),
    'after' for ends (never chop the last line); falls back to the nearest cut."""
    if prefer == "before":
        c = [x for x in cuts if t - tol <= x <= t + 0.05]
        if c:
            return max(c)
    elif prefer == "after":
        c = [x for x in cuts if t - 0.05 <= x <= t + tol]
        if c:
            return min(c)
    best = min(cuts, key=lambda c: abs(c - t)) if cuts else None
    return best if best is not None and abs(best - t) <= tol else None


# --- per-window features -----------------------------------------------------
def events_in(ep: Episode, a: float, b: float) -> list[Event]:
    return [e for e in ep.events if e.start >= a - 0.05 and e.end <= b + 0.05]


def _hits(rx, text: str) -> int:
    return len(rx.findall(text or ""))


def hook_score(ep: Episode, s: float) -> tuple[float, str]:
    """0-10: how hard do the first 3 seconds grab a scroller?"""
    first = [e for e in ep.events if s - 0.05 <= e.start < s + HOOK_WINDOW]
    score, beat = 0.0, ""
    lines = [e for e in first if e.kind == "dialogue"]
    visual = [e for e in first if e.kind in ("action", "camera", "sound")]
    if lines:
        score += 2.5
        ln = lines[0]
        beat = f'{ln.speaker}: "{ln.text}"'
        if re.search(r"[!?]", ln.text): score += 1
        score += min(2.0, 1.0 * _hits(STRONG_RX, ln.text))
        if INTENSE_RX.search(ln.delivery): score += 1
    phys = sum(_hits(PHYSICAL_RX, e.text) for e in visual)
    if phys:
        score += min(4.0, 2.0 * phys)
        beat = beat or next(e.text for e in visual if PHYSICAL_RX.search(e.text))
    if not lines and not phys:
        score -= 1.5 if any(DULL_RX.search(e.text) for e in visual) else 0.5
        beat = beat or (visual[0].text if visual else "(nothing happens)")
    return max(0.0, min(10.0, score * 1.25)), beat


def cliff_score(ep: Episode, s: float, e: float) -> tuple[float, str]:
    """0-10: does it stop on an unresolved beat?"""
    tail = [x for x in ep.events if e - 4.0 <= x.end <= e + 0.05 and x.start >= s]
    score, beat = 0.0, ""
    lines = [x for x in tail if x.kind == "dialogue"]
    if lines:
        last = lines[-1]
        beat = f'{last.speaker}: "{last.text}"'
        if last.text.rstrip().endswith("?"): score += 2.5
        elif "!" in last.text or last.text.rstrip().endswith("..."): score += 1.5
        score += min(2.0, _hits(STRONG_RX, last.text))
        if INTENSE_RX.search(last.delivery): score += 0.5
    snd = [x for x in tail if x.kind == "sound" and CLIFF_RX.search(x.text)]
    if snd:
        score += 2.5
        beat = beat or snd[-1].text
    score += min(2.0, sum(_hits(PHYSICAL_RX, x.text) for x in tail if x.kind != "dialogue"))
    # an answer is coming right after the cut -> the viewer is left hanging
    nxt = [x for x in ep.events if x.kind == "dialogue" and e - 0.05 <= x.start <= e + 2.5]
    if lines and nxt and nxt[0].speaker != lines[-1].speaker: score += 1.5
    if e >= ep.duration - 3.0: score += 2.0      # writers put the episode cliffhanger here
    if not beat and tail: beat = tail[-1].text
    return max(0.0, min(10.0, score * 1.1)), beat


def standalone_score(ep: Episode, s: float, e: float) -> float:
    """0-10: rough guess at whether a stranger can follow it."""
    score = 6.0
    scene_starts = [sc["start"] for sc in ep.scenes]
    if any(abs(s - t) <= 1.0 for t in scene_starts): score += 2.0
    first_line = next((x for x in ep.events if x.kind == "dialogue" and x.start >= s - 0.05), None)
    if first_line and first_line.start < s + 6 and REPLY_RX.search(first_line.text): score -= 2.5
    if any(s + 1 < t < e - 1 for t in scene_starts): score -= 1.5   # jumps location mid-clip
    speakers = {x.speaker for x in events_in(ep, s, e) if x.kind == "dialogue"}
    if len(speakers) > 4: score -= 1.0
    return max(0.0, min(10.0, score))


def intensity_score(ep: Episode, s: float, e: float) -> float:
    ev = events_in(ep, s, e)
    n = sum(1 for x in ev if PHYSICAL_RX.search(x.text) or (x.kind == "dialogue" and (INTENSE_RX.search(x.delivery) or STRONG_RX.search(x.text))))
    return max(0.0, min(10.0, 10.0 * n / max(1.0, (e - s) / 3.0)))


def heuristic_scores(ep: Episode, s: float, e: float) -> dict:
    h, hook_beat = hook_score(ep, s)
    c, cliff_beat = cliff_score(ep, s, e)
    st = standalone_score(ep, s, e)
    it = intensity_score(ep, s, e)
    dur = e - s
    total = 0.35 * h + 0.35 * c + 0.2 * st + 0.1 * it + (0.3 if 25 <= dur <= 45 else 0.0)
    return {"hook": round(h, 1), "cliffhanger": round(c, 1), "standalone": round(st, 1),
            "intensity": round(it, 1), "total": round(total, 2), "hook_beat": hook_beat, "cliff_beat": cliff_beat}


# --- validation ------------------------------------------------------------
def validate_promo(promo: dict, eps: dict[int, Episode], cuts: dict[int, list[float]]) -> tuple[dict | None, str]:
    """Snap every segment to clean cuts and enforce 20-60s total.
    Returns (fixed_promo, "") or (None, reason). Adds 'flags' for anything repaired."""
    flags, segs = [], []
    for seg in promo.get("segments", []):
        n = int(seg["episode"])
        if n not in eps:
            return None, f"episode {n} not in input"
        ep, cp = eps[n], cuts[n]
        try:
            s0, e0 = float(seg["start_s"]), float(seg["end_s"])
        except (KeyError, ValueError):
            return None, "unreadable timestamp"
        s, e = snap(s0, cp, prefer="before"), snap(e0, cp, prefer="after")
        if s is None or e is None:
            return None, f"E{n} {fmt(s0)}-{fmt(e0)}: no clean cut within {SNAP_TOL}s"
        if abs(s - s0) > 0.3 or abs(e - e0) > 0.3:
            flags.append(f"snapped E{n} {fmt(s0)}-{fmt(e0)} -> {fmt(s)}-{fmt(e)}")
        if e <= s:
            return None, f"E{n}: end before start"
        segs.append({"episode": n, "start_s": s, "end_s": e})
    if not segs:
        return None, "no segments"

    total = sum(x["end_s"] - x["start_s"] for x in segs)
    if total > MAX_LEN:
        # trim the START of the last segment's lead-in is risky; trim from the end of the
        # longest segment instead but keep its last line intact by using cut points.
        last = max(segs, key=lambda x: x["end_s"] - x["start_s"])
        over = total - MAX_LEN
        options = [c for c in cuts[last["episode"]] if last["start_s"] + 3 < c <= last["end_s"] - over]
        if not options:
            return None, f"too long ({total:.1f}s) and could not trim"
        flags.append(f"trimmed E{last['episode']} end {fmt(last['end_s'])} -> {fmt(max(options))} (was {total:.1f}s)")
        last["end_s"] = max(options)
    elif total < MIN_LEN:
        last = segs[-1]
        need = MIN_LEN - total
        options = [c for c in cuts[last["episode"]] if c >= last["end_s"] + need]
        if not options or options[0] - last["end_s"] > need + 15:
            return None, f"too short ({total:.1f}s) and could not extend"
        flags.append(f"extended E{last['episode']} end {fmt(last['end_s'])} -> {fmt(options[0])} (was {total:.1f}s)")
        last["end_s"] = options[0]

    total = sum(x["end_s"] - x["start_s"] for x in segs)
    if not (MIN_LEN - 0.01 <= total <= MAX_LEN + 0.01):
        return None, f"length {total:.1f}s outside {MIN_LEN:.0f}-{MAX_LEN:.0f}s after repair"
    # overlapping segments inside one promo
    for a in segs:
        for b in segs:
            if a is not b and a["episode"] == b["episode"] and a["start_s"] < b["end_s"] and b["start_s"] < a["end_s"]:
                return None, "segments overlap"
    for a, b in zip(segs, segs[1:]):
        if a["episode"] > b["episode"] or (a["episode"] == b["episode"] and b["start_s"] < a["start_s"]):
            flags.append("segments play out of story order")
            break
    first = segs[0]
    beats =[x for x in eps[first["episode"]].events
             if x.kind != "camera" and first["start_s"] - 0.05 <= x.start < first["start_s"] + HOOK_WINDOW]
    if not beats:
        flags.append("no dialogue/action in first 3s (hook is visual only)")

    out = dict(promo)
    out["segments"] = [{**x, "start": fmt(x["start_s"]), "end": fmt(x["end_s"])} for x in segs]
    out["duration_s"] = round(total, 2)
    out["flags"] = flags
    return out, ""


# --- no-AI baseline ----------------------------------------------------------
def overlap(a: dict, b: dict) -> float:
    """Fraction of the shorter promo that overlaps the other (same-episode time)."""
    inter = 0.0
    for x in a["segments"]:
        for y in b["segments"]:
            if x["episode"] == y["episode"]:
                inter += max(0.0, min(x["end_s"], y["end_s"]) - max(x["start_s"], y["start_s"]))
    return inter / max(1e-6, min(a["duration_s"], b["duration_s"]))


def baseline_candidates(ep: Episode, cuts: list[float], per_episode: int = 3) -> list[dict]:
    """Slide over every clean (start, end) cut pair of 20-60s, score with rules,
    keep the best non-overlapping windows."""
    scored = []
    for s in cuts:
        for e in cuts:
            if MIN_LEN <= e - s <= MAX_LEN:
                hs = heuristic_scores(ep, s, e)
                scored.append((hs["total"], s, e, hs))
    scored.sort(key=lambda x: -x[0])
    picked: list[dict] = []
    for tot, s, e, hs in scored:
        cand = {"source": "rules", "segments": [{"episode": ep.number, "start_s": s, "end_s": e}], "duration_s": e - s}
        if all(overlap(cand, p) < 0.3 for p in picked):
            cand.update({"title": f"E{ep.number} @ {fmt(s)}", "hook": hs["hook_beat"], "cliffhanger": hs["cliff_beat"],
                         "why": "rule score: strong opening beat + unresolved ending", "rule_scores": hs})
            picked.append(cand)
        if len(picked) >= per_episode:
            break
    return picked

In [ ]:
%%writefile promo_finder/llm.py
"""One tiny interface for every model: complete(system, user) -> text.

Swap models with two settings, no code changes:
    --provider gemini     --model gemini-3.5-flash      (GEMINI_API_KEY)
    --provider groq       --model llama-3.3-70b-versatile (GROQ_API_KEY)
    --provider openai     --model gpt-4o-mini           (OPENAI_API_KEY, OPENAI_BASE_URL optional)
    --provider anthropic  --model claude-sonnet-4-5     (ANTHROPIC_API_KEY)
    --provider openrouter / ollama / any OpenAI-compatible endpoint via --base-url
    --provider mock       (offline, for tests only — fake answers)
Plain HTTP via `requests`, so there is no SDK lock-in.
"""
from __future__ import annotations

import hashlib
import json
import os
import re
import threading
import time
from pathlib import Path

import requests

OPENAI_COMPATIBLE = {
    "openai": ("https://api.openai.com/v1", "OPENAI_API_KEY"),
    "groq": ("https://api.groq.com/openai/v1", "GROQ_API_KEY"),
    "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY"),
    "ollama": ("http://localhost:11434/v1", ""),
}
DEFAULT_MODELS = {
    "gemini": "gemini-3.5-flash",
    "groq": "llama-3.3-70b-versatile",
    "openai": "gpt-4o-mini",
    "openrouter": "meta-llama/llama-3.3-70b-instruct:free",
    "anthropic": "claude-sonnet-4-5",
    "ollama": "llama3.1",
    "mock": "mock",
}


class LLM:
    def __init__(self, provider: str = "gemini", model: str | None = None, base_url: str | None = None,
                 api_key: str | None = None, temperature: float = 0.4, cache_dir: str | None = ".cache",
                 min_interval: float = 0.0, max_retries: int = 5):
        self.provider = provider.lower()
        self.model = model or DEFAULT_MODELS.get(self.provider, "")
        self.base_url = base_url
        self.api_key = api_key
        self.temperature = temperature
        self.cache = Path(cache_dir) if cache_dir else None
        self.min_interval = min_interval      # seconds between calls (free tiers are rate limited)
        self.max_retries = max_retries
        self._last = 0.0
        self._lock = threading.Lock()
        self.calls = 0
        self.cache_hits = 0
        self.chars_in = 0
        self.chars_out = 0
        if self.provider == "mock":
            from .mock import mock_complete
            self._mock = mock_complete

    @property
    def name(self) -> str:
        return f"{self.provider}:{self.model}"

    # ---------------------------------------------------------------------
    def complete(self, system: str, user: str) -> str:
        key = hashlib.sha256(f"{self.name}|{self.temperature}|{system}|{user}".encode()).hexdigest()[:24]
        if self.cache:
            f = self.cache / f"{key}.txt"
            if f.exists():                       # cached reply: count it so cost stats stay true
                text = f.read_text(encoding="utf-8")
                self.calls += 1
                self.cache_hits += 1
                self.chars_in += len(system) + len(user)
                self.chars_out += len(text)
                return text
        with self._lock:                      # pace calls across threads
            wait = self.min_interval - (time.time() - self._last)
            if wait > 0:
                time.sleep(wait)
            self._last = time.time()
        for attempt in range(self.max_retries):
            try:
                text = self._call(system, user)
                break
            except RetryableError as e:
                delay = e.retry_after or min(60, 5 * 2 ** attempt)
                print(f"  [llm] {e} -> retry in {delay:.0f}s")
                time.sleep(delay)
        else:
            raise RuntimeError(f"{self.name}: gave up after {self.max_retries} retries")
        self.calls += 1
        self.chars_in += len(system) + len(user)
        self.chars_out += len(text)
        if self.cache:
            self.cache.mkdir(parents=True, exist_ok=True)
            (self.cache / f"{key}.txt").write_text(text, encoding="utf-8")
        return text

    def complete_json(self, system: str, user: str) -> tuple[dict, str]:
        """Returns (parsed_json, raw_text). One repair retry if the JSON is broken."""
        raw = self.complete(system, user)
        try:
            return extract_json(raw), raw
        except ValueError:
            raw2 = self.complete(system, user + "\n\nYour previous answer was not valid JSON. Reply with ONLY the JSON object.")
            return extract_json(raw2), raw2

    # ---------------------------------------------------------------------
    def _call(self, system: str, user: str) -> str:
        p = self.provider
        if p == "mock":
            return self._mock(system, user)
        if p == "gemini":
            key = self.api_key or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
            url = f"https://generativelanguage.googleapis.com/v1beta/models/{self.model}:generateContent"
            body = {
                "systemInstruction": {"parts": [{"text": system}]},
                "contents": [{"role": "user", "parts": [{"text": user}]}],
                "generationConfig": {"temperature": self.temperature, "responseMimeType": "application/json"},
            }
            r = _post(url, body, {"x-goog-api-key": key or ""})
            try:
                return "".join(part.get("text", "") for part in r["candidates"][0]["content"]["parts"])
            except (KeyError, IndexError):
                raise RuntimeError(f"gemini: unexpected response {json.dumps(r)[:300]}")
        if p == "anthropic":
            key = self.api_key or os.environ.get("ANTHROPIC_API_KEY", "")
            body = {"model": self.model, "max_tokens": 4096, "temperature": self.temperature, "system": system,
                    "messages": [{"role": "user", "content": user}]}
            r = _post("https://api.anthropic.com/v1/messages", body,
                      {"x-api-key": key, "anthropic-version": "2023-06-01"})
            return "".join(b.get("text", "") for b in r.get("content", []))
        # everything else: OpenAI-compatible chat completions
        base, env = OPENAI_COMPATIBLE.get(p, (None, "OPENAI_API_KEY"))
        base = self.base_url or (os.environ.get("OPENAI_BASE_URL") if p == "openai" else None) or base
        base = base or "https://api.openai.com/v1"
        key = self.api_key or (os.environ.get(env, "") if env else "")
        body = {"model": self.model, "temperature": self.temperature,
                "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}
        if p in ("openai", "groq"):
            body["response_format"] = {"type": "json_object"}
        r = _post(base.rstrip("/") + "/chat/completions", body, {"Authorization": f"Bearer {key}"} if key else {})
        return r["choices"][0]["message"]["content"]


class RetryableError(Exception):
    def __init__(self, msg: str, retry_after: float | None = None):
        super().__init__(msg)
        self.retry_after = retry_after


def _post(url: str, body: dict, headers: dict) -> dict:
    try:
        r = requests.post(url, json=body, headers={"Content-Type": "application/json", **headers}, timeout=600)
    except requests.RequestException as e:
        raise RetryableError(f"network error: {e}")
    if r.status_code in (429, 500, 502, 503, 504):
        ra = r.headers.get("retry-after")
        raise RetryableError(f"HTTP {r.status_code}", float(ra) if ra and ra.replace('.', '', 1).isdigit() else None)
    if r.status_code >= 400:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:400]}")
    return r.json()


def extract_json(text: str) -> dict:
    """Pull the first JSON object out of a model reply (handles ```json fences and chatter)."""
    t = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        pass
    start = t.find("{")
    while start != -1:
        depth, in_str, esc = 0, False, False
        for i in range(start, len(t)):
            ch = t[i]
            if in_str:
                if esc:
                    esc = False
                elif ch == "\\":
                    esc = True
                elif ch == '"':
                    in_str = False
                continue
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(t[start:i + 1])
                    except json.JSONDecodeError:
                        break
        start = t.find("{", start + 1)
    raise ValueError("no JSON object found in model reply")

In [ ]:
%%writefile promo_finder/prompts.py
"""All prompts in one place so they are easy to read and tune."""

SYSTEM = """You are a senior promo editor for vertical microdrama apps like ReelShort and DramaBox.
You pick 20-60 second clips from a show that make a stranger scrolling TikTok/Reels stop, watch, and need the next episode.
A great promo:
- HOOK: something in the FIRST 3 SECONDS that stops a scroll - a slap, a reveal, a shocking or humiliating line, a threat. Never an establishing shot or a slow walk-in.
- CLIFFHANGER: it CUTS at an unresolved moment - right after a question, accusation, threat or reveal, BEFORE the answer or the reaction resolves it.
- STANDALONE: someone who has never seen the show understands who is wronging whom within seconds.
You only use the material you are given. You reply with JSON only."""

EPISODE = """TASK: EPISODE
Show: {show}
Premise (from episode 1): {premise}
Episode {num} logline: {logline}

Below is the timestamped screenplay. Every line has an ID (#n). CAMERA lines are shots, ACTION/SOUND are what we see/hear, NAME lines are dialogue.
{transcript}

Find up to {k} promos in THIS episode, best first. Return fewer (even zero) if the episode has no strong moment - do not pad.
Rules:
- Each promo is 1 to 3 segments from this episode, in playback order. Total length 20-60 seconds (check with the timestamps).
- Give each segment as from_id/to_id (the IDs of its first and last line). Start on a CAMERA line where possible. Never start or end in the middle of someone's sentence.
- The line at from_id of the first segment must BE the hook (within 3 seconds).
- The line at to_id of the last segment must be the cliffhanger - cut before the answer.
- Score honestly from 1-10; a 9-10 should be rare.

Also list up to 4 key_moments: short beats (3-15s) from this episode that would work inside a cross-episode montage (a big hook, reveal, or cliffhanger).

JSON format:
{{"promos": [{{"title": "short name", "segments": [{{"from_id": 12, "to_id": 40}}],
  "hook": "what happens in the first 3 seconds", "cliffhanger": "the unresolved beat at the cut",
  "why": "one line on why it works", "scores": {{"hook": 7, "cliffhanger": 8, "standalone": 6}},
  "spoiler_risk": "low|medium|high"}}],
 "key_moments": [{{"from_id": 3, "to_id": 9, "type": "hook|reveal|cliffhanger|action", "summary": "one line"}}]}}"""

MONTAGE = """TASK: MONTAGE
Show: {show}
Premise (from episode 1): {premise}

Key moments from across the season (ID, episode, time, length, type, summary, dialogue):
{moments}

Build up to {k} cross-episode promos by joining 2-4 of these moments. Each must:
- total 20-60 seconds (add up the lengths),
- open on the strongest hook moment, end on a cliffhanger moment,
- tell ONE clear mini-story a stranger can follow (setup -> escalation -> unresolved turn), not a random highlight reel,
- avoid giving away the biggest late-season twists unless the show title already gives it away.
Only use the IDs listed. JSON only:
{{"montages": [{{"title": "short name", "moment_ids": ["M3_1", "M9_2"], "hook": "...", "cliffhanger": "...",
  "why": "one line", "scores": {{"hook": 7, "cliffhanger": 8, "standalone": 6}}, "spoiler_risk": "low|medium|high"}}]}}"""

JUDGE = """TASK: JUDGE
You are checking promos another editor picked. You see exactly what a viewer would see and hear, nothing else.
Score each one strictly from 1-10:
- hook: do the FIRST 3 SECONDS (marked >>) stop a scroll?
- cliffhanger: do the LAST SECONDS (marked <<) leave something clearly unresolved?
- standalone: would a stranger understand it with no context?
- spoiler_risk: low|medium|high
- note: the single biggest problem, or "ok".

{promos}

JSON only: {{"ratings": [{{"id": "P1", "hook": 6, "cliffhanger": 7, "standalone": 5, "spoiler_risk": "low", "note": "..."}}]}}"""

EPISODE_BATCH = """TASK: EPISODE
Show: {show}
Premise (from episode 1): {premise}

Below are {n} episodes. Treat each episode on its own. Every line has an ID (#n) that is unique only WITHIN its episode.
CAMERA lines are shots, ACTION/SOUND are what we see/hear, NAME lines are dialogue.
{blocks}

For EACH episode, find up to {k} promos inside that episode, best first. Return fewer (even zero) if it has no strong moment - do not pad.
Rules:
- Each promo is 1 to 3 segments from that one episode, in playback order. Total length 20-60 seconds (check with the timestamps).
- Give each segment as from_id/to_id (IDs of its first and last line, from that episode). Start on a CAMERA line where possible. Never start or end in the middle of someone's sentence.
- The line at from_id of the first segment must BE the hook (within 3 seconds).
- The line at to_id of the last segment must be the cliffhanger - cut before the answer.
- Score honestly from 1-10; a 9-10 should be rare.
Also list up to 4 key_moments per episode: short beats (3-15s) that would work inside a cross-episode montage.

JSON format:
{{"episodes": [{{"episode": 3,
  "promos": [{{"title": "short name", "segments": [{{"from_id": 12, "to_id": 40}}],
    "hook": "what happens in the first 3 seconds", "cliffhanger": "the unresolved beat at the cut",
    "why": "one line on why it works", "scores": {{"hook": 7, "cliffhanger": 8, "standalone": 6}},
    "spoiler_risk": "low|medium|high"}}],
  "key_moments": [{{"from_id": 3, "to_id": 9, "type": "hook|reveal|cliffhanger|action", "summary": "one line"}}]}}]}}"""

In [ ]:
%%writefile promo_finder/pipeline.py
"""End-to-end: screenplays -> ranked promos + raw outputs.

    rules (always)  ->  AI per episode  ->  AI cross-episode montages
         \\                    |                     /
          +--> code validation (snap to clean cuts, 20-60s) --> AI judge (blind) --> rank --> files
"""
from __future__ import annotations

import csv
import json
import re
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from . import prompts
from .llm import LLM
from .parser import Episode, load_show, fmt
from .rules import (baseline_candidates, cut_points, heuristic_scores, hook_score, cliff_score,
                    standalone_score, overlap, validate_promo, HOOK_WINDOW)


# --------------------------------------------------------------------------
def numbered_transcript(ep: Episode) -> str:
    return "\n".join(f"#{e.idx} {e.line()}" for e in ep.events)


def ids_to_segment(ep: Episode, a, b) -> dict:
    a, b = int(a), int(b)
    if not (0 <= a < len(ep.events) and 0 <= b < len(ep.events)):
        raise ValueError(f"line id out of range ({a},{b})")
    if b < a:
        a, b = b, a
    seg_events = ep.events[a:b + 1]
    return {"episode": ep.number, "start_s": ep.events[a].start, "end_s": max(e.end for e in seg_events)}


def rule_view(promo: dict, eps: dict[int, Episode]) -> dict:
    """Rule scores for any promo (multi-segment: hook from first segment, cliffhanger from last)."""
    segs = promo["segments"]
    f, l = segs[0], segs[-1]
    if len(segs) == 1:
        return heuristic_scores(eps[f["episode"]], f["start_s"], f["end_s"])
    h, hb = hook_score(eps[f["episode"]], f["start_s"])
    c, cb = cliff_score(eps[l["episode"]], l["start_s"], l["end_s"])
    st = sum(standalone_score(eps[s["episode"]], s["start_s"], s["end_s"]) for s in segs) / len(segs)
    return {"hook": round(h, 1), "cliffhanger": round(c, 1), "standalone": round(st, 1), "intensity": None,
            "total": round(0.35 * h + 0.35 * c + 0.2 * st + 0.1 * 5, 2), "hook_beat": hb, "cliff_beat": cb}


def viewer_script(promo: dict, eps: dict[int, Episode], kinds=("action", "dialogue", "sound", "camera")) -> str:
    """What the viewer sees/hears, in order, with >> for the first 3s and << for the last 4s."""
    out, clock = [], 0.0
    total = promo["duration_s"]
    for seg in promo["segments"]:
        ep = eps[seg["episode"]]
        out.append(f"--- E{ep.number} {seg['start']}-{seg['end']} ---")
        for e in ep.events:
            if e.kind in kinds and e.start >= seg["start_s"] - 0.05 and e.start < seg["end_s"] - 0.05:
                t = clock + (e.start - seg["start_s"])
                mark = ">>" if t < HOOK_WINDOW else ("<<" if t >= total - 4.0 else "  ")
                out.append(f"{mark} {e.line()}")
        clock += seg["end_s"] - seg["start_s"]
    return "\n".join(out)


# --------------------------------------------------------------------------
class PromoFinder:
    def __init__(self, llm: LLM | None = None, per_episode: int = 3, montages: int = 5, judge: bool = True,
                 judge_top: int = 25, workers: int = 1, batch: int = 1, judge_batch: int = 5, log=print):
        self.llm, self.k, self.n_montages, self.judge, self.judge_top, self.log = llm, per_episode, montages, judge, judge_top, log
        self.workers = max(1, workers)
        self.batch = max(1, batch)              # episodes per AI call (fewer calls on tight free quotas)
        self.judge_batch = max(1, judge_batch)

    def run(self, input_dir: str, out_dir: str, episodes: list[int] | None = None, show_name: str | None = None) -> dict:
        t0 = time.time()
        out = Path(out_dir)
        (out / "raw_llm").mkdir(parents=True, exist_ok=True)
        eps_list = load_show(input_dir)
        if episodes:
            eps_list = [e for e in eps_list if e.number in set(episodes)]
        if not eps_list:
            raise SystemExit(f"No screenplays found in {input_dir}")
        eps = {e.number: e for e in eps_list}
        cuts = {n: cut_points(e) for n, e in eps.items()}
        show = show_name or Path(input_dir).name
        premise = eps_list[0].logline
        self.log(f"Loaded {len(eps)} episodes ({sum(e.duration for e in eps_list)/60:.1f} min of screen time)")

        promos, rejected = [], []

        # 1) rules baseline (always runs; also the fallback when no API key)
        for ep in eps_list:
            for c in baseline_candidates(ep, cuts[ep.number], self.k):
                v, why = validate_promo(c, eps, cuts)
                (promos.append(v) if v else rejected.append({"source": "rules", "reason": why, "promo": c}))

        # 2) AI per episode
        moments = {}
        if self.llm:
            def ask(group):
                nums = ", ".join(str(e.number) for e in group)
                self.log(f"  AI: episode(s) {nums}")
                if len(group) == 1:
                    ep = group[0]
                    user = prompts.EPISODE.format(show=show, premise=premise, num=ep.number, logline=ep.logline,
                                                  transcript=numbered_transcript(ep), k=self.k)
                else:
                    blocks = "\n\n".join(f"===== EPISODE {e.number} =====\nLogline: {e.logline}\n{numbered_transcript(e)}" for e in group)
                    user = prompts.EPISODE_BATCH.format(show=show, premise=premise, n=len(group), blocks=blocks, k=self.k)
                try:
                    return self.llm.complete_json(prompts.SYSTEM, user)
                except Exception as e:  # keep going; report it
                    return e
            groups = [eps_list[i:i + self.batch] for i in range(0, len(eps_list), self.batch)]
            with ThreadPoolExecutor(max_workers=self.workers) as pool:   # episodes are independent
                answers = list(pool.map(ask, groups))
            per_ep_answers = []
            for group, ans in zip(groups, answers):
                tag = f"{group[0].number:03d}" + (f"-{group[-1].number:03d}" if len(group) > 1 else "")
                if isinstance(ans, Exception):
                    for ep in group:
                        rejected.append({"source": "ai", "episode": ep.number, "reason": f"LLM error: {ans}"})
                    continue
                data, raw = ans
                if isinstance(data, list):           # some models drop the outer object
                    data = {"episodes": data} if len(group) > 1 else (data[0] if data else {})
                (out / "raw_llm" / f"episode_{tag}.json").write_text(raw, encoding="utf-8")
                if "episodes" in data:
                    by_num = {}
                    for d in data["episodes"]:
                        m = re.search(r"\d+", str(d.get("episode", ""))) if isinstance(d, dict) else None
                        if m:
                            by_num[int(m.group())] = d
                    for ep in group:
                        per_ep_answers.append((ep, by_num.get(ep.number, {})))
                else:
                    per_ep_answers.append((group[0], data))
            for ep, data in per_ep_answers:
                for p in data.get("promos", [])[: self.k]:
                    try:
                        segs = [ids_to_segment(ep, s["from_id"], s["to_id"]) for s in p.get("segments", [])]
                    except (KeyError, ValueError, TypeError) as e:
                        rejected.append({"source": "ai", "episode": ep.number, "reason": f"bad segment ids: {e}", "promo": p})
                        continue
                    cand = {**p, "source": "ai", "segments": segs}
                    v, why = validate_promo(cand, eps, cuts)
                    (promos.append(v) if v else rejected.append({"source": "ai", "episode": ep.number, "reason": why, "promo": p}))
                for j, m in enumerate(data.get("key_moments", [])[:4], 1):
                    try:
                        seg = ids_to_segment(ep, m["from_id"], m["to_id"])
                    except (KeyError, ValueError, TypeError):
                        continue
                    if 2 <= seg["end_s"] - seg["start_s"] <= 20:
                        moments[f"M{ep.number}_{j}"] = {**m, **seg}

        # 3) AI cross-episode montages
        if self.llm and self.n_montages and len(moments) >= 2:
            self.log(f"  AI: montages from {len(moments)} key moments")
            lines = []
            for mid, m in moments.items():
                ep = eps[m["episode"]]
                said = " / ".join(f'{e.speaker}: {e.text}' for e in ep.events
                                  if e.kind == "dialogue" and m["start_s"] - 0.05 <= e.start < m["end_s"])
                lines.append(f"{mid}: E{m['episode']} [{fmt(m['start_s'])}-{fmt(m['end_s'])}] "
                             f"({m['end_s'] - m['start_s']:.1f}s) {m.get('type', '')}: {m.get('summary', '')} | {said[:300]}")
            try:
                data, raw = self.llm.complete_json(prompts.SYSTEM, prompts.MONTAGE.format(
                    show=show, premise=premise, moments="\n".join(lines), k=self.n_montages))
                (out / "raw_llm" / "montages.json").write_text(raw, encoding="utf-8")
                if isinstance(data, list):
                    data = {"montages": data}
                for p in data.get("montages", [])[: self.n_montages]:
                    ids = [i for i in p.get("moment_ids", []) if i in moments]
                    if len(ids) < 2:
                        rejected.append({"source": "ai-montage", "reason": "unknown or too few moment ids", "promo": p})
                        continue
                    segs = [{k: moments[i][k] for k in ("episode", "start_s", "end_s")} for i in ids]
                    v, why = validate_promo({**p, "source": "ai-montage", "segments": segs, "moment_ids": ids}, eps, cuts)
                    (promos.append(v) if v else rejected.append({"source": "ai-montage", "reason": why, "promo": p}))
            except Exception as e:
                rejected.append({"source": "ai-montage", "reason": f"LLM error: {e}"})

        # 4) rule scores for everything + agreement between AI and rules
        for p in promos:
            p["rule_scores"] = rule_view(p, eps)
        rule_ps = [p for p in promos if p["source"] == "rules"]
        for p in promos:
            if p["source"] == "ai":
                p["also_found_by_rules"] = any(overlap(p, r) >= 0.5 for r in rule_ps)

        # 5) preliminary score, then blind AI judge. Self-scores are inflated (the picker rates its own
        #    picks 8-9), so keep judging until every promo in the top `judge_top` has a blind score.
        last_ep = max(eps)
        for p in promos:
            p["season_position"] = round(p["segments"][0]["episode"] / last_ep, 2)
            p["final_score"] = self._score(p)
        promos.sort(key=lambda p: -p["final_score"])
        if self.llm and self.judge:
            raws = []
            for _ in range(4):
                todo = [p for p in promos[: self.judge_top] if not p.get("judge")]
                if not todo:
                    break
                raws += self._judge(todo, eps, self.judge_batch)
                for p in promos:
                    p["final_score"] = self._score(p)
                promos.sort(key=lambda p: -p["final_score"])
            (out / "raw_llm" / "judge.json").write_text(json.dumps(raws, indent=2, ensure_ascii=False), encoding="utf-8")

        # 6) when AI and rules found the same moment, keep the AI version (better hook/why text)
        #    and note the agreement; then rank with diversity
        ai_ps = [p for p in promos if p["source"] != "rules"]
        promos = [p for p in promos if p["source"] != "rules" or not any(overlap(p, q) >= 0.5 for q in ai_ps)]

        def dup(p, q):  # montages only compete with montages
            if (p["source"] == "ai-montage") != (q["source"] == "ai-montage"):
                return False
            return overlap(p, q) >= 0.5

        ranked, per_ep = [], {}
        for p in sorted(promos, key=lambda p: -p["final_score"]):
            if any(dup(p, q) for q in ranked):
                p["dropped_as_duplicate"] = True
                continue
            first_ep = p["segments"][0]["episode"]
            if len(ranked) < 10 and per_ep.get(first_ep, 0) >= 2:
                continue
            per_ep[first_ep] = per_ep.get(first_ep, 0) + 1
            ranked.append(p)
        leftovers = [p for p in promos if p not in ranked and not p.get("dropped_as_duplicate")]
        ranked += sorted(leftovers, key=lambda p: -p["final_score"])
        for i, p in enumerate(ranked, 1):
            p["rank"] = i
            p["promo_id"] = f"P{i:03d}"

        meta = self._write(out, ranked, rejected, eps, show, time.time() - t0)
        self.log(f"Done: {len(ranked)} promos, {len(rejected)} rejected -> {out}")
        return meta

    # ----------------------------------------------------------------------
    @staticmethod
    def _score(p: dict) -> float:
        r = p["rule_scores"]
        ai = p.get("judge") or p.get("scores")
        if ai:
            q = 0.4 * float(ai.get("hook", 5)) + 0.35 * float(ai.get("cliffhanger", 5)) + 0.25 * float(ai.get("standalone", 5))
            score = 0.8 * q + 0.2 * r["total"]
        else:
            score = r["total"]
        risk = ((p.get("judge") or {}).get("spoiler_risk") or p.get("spoiler_risk") or "low").lower()
        score -= {"high": 1.0, "medium": 0.3}.get(risk, 0.0)
        first_ep = p["segments"][0]["episode"]
        score += 0.5 if first_ep <= 3 else (0.25 if first_ep <= 10 else 0.0)   # early = matches what a new viewer sees first
        if p.get("season_position", 0) > 0.8:
            score -= 0.6                                                       # final act resolves the main conflict
        if p.get("also_found_by_rules"):
            score += 0.3                                                       # two independent pickers agree
        return round(score, 2)

    def _judge(self, top: list[dict], eps, batch: int = 5) -> list:
        self.log(f"  AI judge: {len(top)} promos")
        raws = []
        for i in range(0, len(top), batch):
            chunk = top[i:i + batch]
            blocks = []
            for j, p in enumerate(chunk, 1):
                blocks.append(f"### P{j} ({p['duration_s']:.0f}s)\n{viewer_script(p, eps, kinds=('action', 'dialogue', 'sound'))}")
            try:
                data, raw = self.llm.complete_json(prompts.SYSTEM, prompts.JUDGE.format(promos="\n\n".join(blocks)))
                raws.append(data)
            except Exception as e:
                self.log(f"  judge failed: {e}")
                continue
            for r in (data if isinstance(data, list) else data.get("ratings", [])):
                try:
                    k = int(str(r.get("id", "")).lstrip("P")) - 1
                    chunk[k]["judge"] = {x: r.get(x) for x in ("hook", "cliffhanger", "standalone", "spoiler_risk", "note")}
                except (ValueError, IndexError):
                    pass
        return raws

    def _write(self, out: Path, ranked, rejected, eps, show, secs) -> dict:
        seg_str = lambda p: " | ".join(f"E{s['episode']} {s['start']}-{s['end']}" for s in p["segments"])
        clean = lambda p: {k: v for k, v in p.items() if k not in ("dropped_as_duplicate",)}
        (out / "promos.json").write_text(json.dumps([clean(p) for p in ranked], indent=2, ensure_ascii=False), encoding="utf-8")
        (out / "rejected.json").write_text(json.dumps(rejected, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

        cols = ["rank", "promo_id", "source", "segments", "duration_s", "hook", "cliffhanger", "why", "final_score",
                "ai_hook", "ai_cliffhanger", "ai_standalone", "judge_hook", "judge_cliffhanger", "judge_standalone",
                "judge_note", "rule_total", "spoiler_risk", "also_found_by_rules", "flags"]
        with open(out / "promos.csv", "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=cols)
            w.writeheader()
            for p in ranked:
                s, j = p.get("scores") or {}, p.get("judge") or {}
                w.writerow({"rank": p["rank"], "promo_id": p["promo_id"], "source": p["source"], "segments": seg_str(p),
                            "duration_s": p["duration_s"], "hook": p.get("hook", ""), "cliffhanger": p.get("cliffhanger", ""),
                            "why": p.get("why", ""), "final_score": p["final_score"], "ai_hook": s.get("hook"),
                            "ai_cliffhanger": s.get("cliffhanger"), "ai_standalone": s.get("standalone"),
                            "judge_hook": j.get("hook"), "judge_cliffhanger": j.get("cliffhanger"),
                            "judge_standalone": j.get("standalone"), "judge_note": j.get("note"),
                            "rule_total": p["rule_scores"]["total"], "spoiler_risk": j.get("spoiler_risk") or p.get("spoiler_risk", ""),
                            "also_found_by_rules": p.get("also_found_by_rules", ""), "flags": "; ".join(p.get("flags", []))})

        # sheet for a human editor to grade the top picks (the real quality check)
        with open(out / "review_sheet.csv", "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["rank", "promo_id", "segments", "hook", "cliffhanger", "would_post(y/n)", "hook_ok(y/n)",
                        "cliffhanger_ok(y/n)", "standalone_ok(y/n)", "timing_fix", "notes"])
            for p in ranked[:20]:
                w.writerow([p["rank"], p["promo_id"], seg_str(p), p.get("hook", ""), p.get("cliffhanger", ""), "", "", "", "", "", ""])

        ai = [p for p in ranked if p["source"] == "ai"]
        meta = {
            "show": show, "episodes": len(eps), "model": self.llm.name if self.llm else "rules-only (no AI)",
            "runtime_s": round(secs, 1), "promos": len(ranked),
            "by_source": {s: sum(1 for p in ranked if p["source"] == s) for s in ("rules", "ai", "ai-montage")},
            "rejected": len(rejected),
            "repaired_by_code": sum(1 for p in ranked if p.get("flags")),
            "ai_rules_agreement": round(sum(1 for p in ai if p.get("also_found_by_rules")) / len(ai), 2) if ai else None,
            "llm_calls": self.llm.calls if self.llm else 0,
            "llm_calls_served_from_cache": self.llm.cache_hits if self.llm else 0,
            "approx_tokens_in": (self.llm.chars_in // 4) if self.llm else 0,
            "approx_tokens_out": (self.llm.chars_out // 4) if self.llm else 0,
        }
        (out / "run_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

        # human-readable report
        md = [f"# Promo picks - {show}", "",
              f"Model: `{meta['model']}` | episodes: {meta['episodes']} | promos: {meta['promos']} "
              f"(rules {meta['by_source']['rules']}, AI {meta['by_source']['ai']}, montages {meta['by_source']['ai-montage']}) "
              f"| rejected: {meta['rejected']} | repaired by code: {meta['repaired_by_code']}", "",
              "## Post first (top 10)", "",
              "| # | Segments | Len | Score | Hook | Cliffhanger |", "|---|---|---|---|---|---|"]
        for p in ranked[:10]:
            md.append(f"| {p['rank']} | {seg_str(p)} | {p['duration_s']:.0f}s | {p['final_score']} | "
                      f"{str(p.get('hook', '')).replace('|', '/')[:90]} | {str(p.get('cliffhanger', '')).replace('|', '/')[:90]} |")
        md += ["", "## Details (top 15)", ""]
        for p in ranked[:15]:
            j = p.get("judge") or {}
            md += [f"### {p['rank']}. {p.get('title', p['promo_id'])}  ({p['source']}, {p['duration_s']:.0f}s, score {p['final_score']})",
                   f"- **Segments:** {seg_str(p)}",
                   f"- **Hook:** {p.get('hook', '')}",
                   f"- **Cliffhanger:** {p.get('cliffhanger', '')}",
                   f"- **Why:** {p.get('why', '')}"]
            if j:
                md.append(f"- **Blind judge:** hook {j.get('hook')}, cliffhanger {j.get('cliffhanger')}, standalone {j.get('standalone')} - {j.get('note')}")
            if p.get("flags"):
                md.append(f"- **Code fixes/flags:** {'; '.join(p['flags'])}")
            md += ["", "```", viewer_script(p, eps, kinds=("action", "dialogue", "sound")), "```", ""]
        if rejected:
            md += ["## Rejected by validation", ""] + [f"- {r.get('source')} E{r.get('episode', '?')}: {r['reason']}" for r in rejected[:40]]
        (out / "report.md").write_text("\n".join(md), encoding="utf-8")
        return meta

In [ ]:
%%writefile promo_finder/mock.py
"""Fake model for offline tests: exercises the whole AI code path without a key.
Its picks are deliberately naive - never use mock output as real results."""
import json
import re

LINE = re.compile(r"^#(\d+) \[(\d+):(\d+\.\d+)-(\d+):(\d+\.\d+)\] (\w[^:]*):(.*)$", re.M)


def mock_complete(system: str, user: str) -> str:
    if "TASK: EPISODE" in user:
        rows = [(int(i), int(a) * 60 + float(b), int(c) * 60 + float(d), who, txt) for i, a, b, c, d, who, txt in LINE.findall(user)]
        promos, moments = [], []
        starts = [r for r in rows if r[3] not in ("CAMERA", "ACTION", "SOUND") and re.search(r"[!?]", r[4])]
        for r in starts[:: max(1, len(starts) // 2 or 1)][:2]:
            end = next((x for x in rows if x[2] >= r[1] + 32), rows[-1])
            promos.append({"title": f"mock {r[0]}", "segments": [{"from_id": r[0], "to_id": end[0]}],
                           "hook": r[4].strip()[:80], "cliffhanger": end[4].strip()[:80], "why": "mock",
                           "scores": {"hook": 6, "cliffhanger": 6, "standalone": 6}, "spoiler_risk": "low"})
            moments.append({"from_id": r[0], "to_id": r[0] + 3, "type": "hook", "summary": r[4].strip()[:60]})
        promos.append({"title": "mock bad ids", "segments": [{"from_id": 99999, "to_id": 100000}]})
        return json.dumps({"promos": promos, "key_moments": moments})
    if "TASK: MONTAGE" in user:
        ids = re.findall(r"^(M\d+_\d+):", user, re.M)
        out = [{"title": "mock montage", "moment_ids": ids[i:i + 3], "hook": "mock", "cliffhanger": "mock", "why": "mock",
                "scores": {"hook": 6, "cliffhanger": 6, "standalone": 5}, "spoiler_risk": "medium"} for i in (0, 6, 12) if len(ids) > i + 2]
        return "Sure! Here you go:\n```json\n" + json.dumps({"montages": out}) + "\n```"
    if "TASK: JUDGE" in user:
        n = len(re.findall(r"^### P\d+", user, re.M))
        return json.dumps({"ratings": [{"id": f"P{i}", "hook": 5, "cliffhanger": 5, "standalone": 5,
                                        "spoiler_risk": "low", "note": "mock"} for i in range(1, n + 1)]})
    return "{}"

In [ ]:
%%writefile find_promos.py
#!/usr/bin/env python3
"""Find and rank promo clips in a microdrama screenplay folder.

Examples
  python find_promos.py --input ./scripts --out ./outputs --provider none          # rules only, no key
  python find_promos.py --input ./scripts --out ./outputs --provider gemini         # needs GEMINI_API_KEY
  python find_promos.py --input ./scripts --out ./outputs --provider groq --model llama-3.3-70b-versatile
  python find_promos.py --input ./scripts --out ./outputs --provider openai --base-url https://YOUR-API/v1 --model YOUR-MODEL
"""
import argparse

from promo_finder.llm import LLM
from promo_finder.pipeline import PromoFinder


def parse_eps(s: str | None):
    if not s:
        return None
    out = []
    for part in s.split(","):
        a, _, b = part.partition("-")
        out += list(range(int(a), int(b or a) + 1))
    return out


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--input", required=True, help="folder (or unzipped pack) of episode .txt screenplays")
    ap.add_argument("--out", default="outputs", help="output folder")
    ap.add_argument("--provider", default="gemini", help="gemini | groq | openai | openrouter | anthropic | ollama | mock | none")
    ap.add_argument("--model", default=None, help="model name (default depends on provider)")
    ap.add_argument("--base-url", default=None, help="for any OpenAI-compatible API")
    ap.add_argument("--episodes", default=None, help="e.g. 1-10 or 1,4,7 (default: all)")
    ap.add_argument("--per-episode", type=int, default=3, help="max promos per episode")
    ap.add_argument("--montages", type=int, default=5, help="max cross-episode promos (0 = off)")
    ap.add_argument("--no-judge", action="store_true", help="skip the blind AI re-scoring pass")
    ap.add_argument("--rpm", type=float, default=8, help="max AI calls per minute (free tiers are rate limited)")
    ap.add_argument("--workers", type=int, default=1, help="parallel AI calls for episodes (raise if your quota allows)")
    ap.add_argument("--batch", type=int, default=1, help="episodes per AI call (e.g. 5 to stay inside a 20-requests/day free tier)")
    ap.add_argument("--judge-batch", type=int, default=5, help="promos per judge call")
    ap.add_argument("--show", default=None, help="show name for prompts/report")
    a = ap.parse_args()

    llm = None
    if a.provider.lower() != "none":
        llm = LLM(a.provider, a.model, base_url=a.base_url, min_interval=60.0 / a.rpm if a.rpm else 0.0,
                  cache_dir=".cache" if a.provider != "mock" else None)
    PromoFinder(llm, per_episode=a.per_episode, montages=a.montages, judge=not a.no_judge,
                workers=a.workers, batch=a.batch, judge_batch=a.judge_batch).run(
        a.input, a.out, episodes=parse_eps(a.episodes), show_name=a.show)


if __name__ == "__main__":
    main()

## 2. Upload the screenplays
Upload the input-pack **.zip** (or the .txt files). Files stay inside this Colab session.

In [ ]:
import os, zipfile, glob
from google.colab import files
os.makedirs("input", exist_ok=True)
for name, data in files.upload().items():
    path = os.path.join("input", name)
    open(path, "wb").write(data)
    if name.lower().endswith(".zip"):
        zipfile.ZipFile(path).extractall("input")
print(len(glob.glob("input/**/*.txt", recursive=True)), "screenplay files found")

## 3. Pick the model
Change `PROVIDER` / `MODEL` to swap models, no code changes. Options: `gemini`, `groq`, `openai`, `openrouter`, `anthropic`, or any OpenAI-compatible API via `BASE_URL`. Use `none` to run rules only.

In [ ]:
PROVIDER = "gemini"            #@param ["gemini", "groq", "openai", "openrouter", "anthropic", "none"]
MODEL = "gemini-3.5-flash-lite"  #@param {type:"string"}
BASE_URL = ""                  #@param {type:"string"}
RPM = 10                       #@param {type:"number"}
BATCH = 5                      #@param {type:"number"}
EPISODES = ""                  #@param {type:"string"}
SHOW = ""                      #@param {type:"string"}

import os, getpass
ENV = {"gemini": "GEMINI_API_KEY", "groq": "GROQ_API_KEY", "openai": "OPENAI_API_KEY",
       "openrouter": "OPENROUTER_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}.get(PROVIDER)
if ENV and not os.environ.get(ENV):
    try:
        from google.colab import userdata
        os.environ[ENV] = userdata.get(ENV)          # Colab "Secrets" panel (key icon on the left)
    except Exception:
        os.environ[ENV] = getpass.getpass(f"{ENV}: ")
print("Using", PROVIDER, MODEL)

In [ ]:
#@title (optional) list the Gemini models your key can use
import requests, os
if PROVIDER == "gemini":
    r = requests.get("https://generativelanguage.googleapis.com/v1beta/models", headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]}).json()
    print([m["name"].split("/")[-1] for m in r.get("models", []) if "generateContent" in m.get("supportedGenerationMethods", [])])

## 4. Run
Rules-only first (seconds), then the AI run (5 episodes per call: ~17 calls, fits the 20-requests/day free tier; ~1 min).

In [ ]:
!python find_promos.py --input input --out outputs/rules_only --provider none --show "$SHOW"

In [ ]:
import shlex, subprocess
cmd = f"python find_promos.py --input input --out outputs/ai_{PROVIDER} --provider {PROVIDER} --model {shlex.quote(MODEL)} --rpm {RPM} --batch {BATCH} --judge-batch 13 " + (f" --show {shlex.quote(SHOW)}" if SHOW else "")
if BASE_URL: cmd += f" --base-url {shlex.quote(BASE_URL)}"
if EPISODES: cmd += f" --episodes {EPISODES}"
if PROVIDER != "none":
    !{cmd}

## 5. Results

In [ ]:
import pandas as pd, json
out = f"outputs/ai_{PROVIDER}" if PROVIDER != "none" else "outputs/rules_only"
print(json.dumps(json.load(open(f"{out}/run_meta.json")), indent=2))
df = pd.read_csv(f"{out}/promos.csv")
pd.set_option("display.max_colwidth", 80)
df[["rank", "source", "segments", "duration_s", "hook", "cliffhanger", "why", "final_score"]].head(15)

In [ ]:
from IPython.display import Markdown
Markdown(open(f"{out}/report.md").read()[:20000])

In [ ]:
!cd outputs && zip -qr ../promo_outputs.zip . && cd ..
files.download("promo_outputs.zip")